# Phase 6 — Stronger video classifiers

Run on Kaggle with **T4 x2**, Internet enabled, and W&B secret `WANDB_API_KEY`. Attach: (1) RWF-2000, (2) the Phase 3 interaction crop-cache dataset, and (3) the Phase 5 output dataset containing `x3d_s_rwf2000_late_fusion/best.pt`. The notebook rebuilds temporary paired tensors under `/kaggle/temp`, trains X3D-M late fusion and optional Swin3D-T, and writes run histories, evaluations, comparisons, and cache reports under `/kaggle/working/experiments/phase6`, with reusable checkpoints under `/kaggle/working/model_checkpoints/phase6`.

The Phase 5 X3D-S late-fusion model is the baseline. A final comparison uses the same 400 RWF-2000 test clips and batch-one single-GPU inference timing.

## 1. Clone the repository, install Kaggle-only dependencies, and inspect GPUs

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception as exc:
    print('W&B secret unavailable; logger fallback will be used:', exc)

REPO_URL = 'https://github.com/Ayaan-Ali-Khan/Violence-Detection.git'
REPO_DIR = Path('/kaggle/working/Violence-Detection')
GIT_REF = 'main'  # Pin a commit SHA for final report runs.
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', GIT_REF], check=True)
if GIT_REF in {'main', 'master'}:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', GIT_REF], check=True)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'src')
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-kaggle.txt'], check=True)
GPU_COUNT = torch.cuda.device_count()
print('PyTorch:', torch.__version__, '| visible GPUs:', GPU_COUNT)
for gpu_id in range(GPU_COUNT):
    print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}')
if GPU_COUNT < 1:
    raise RuntimeError('Select a Kaggle GPU accelerator before continuing.')
if GPU_COUNT != 2:
    print('Select T4 x2 to match the planned training setup.')


## 2. Locate the datasets, paired crop cache, and Phase 5 baseline checkpoint

In [ ]:
import pandas as pd
import zipfile

INPUT_ROOT = Path('/kaggle/input')
print('Kaggle input mounts:', *INPUT_ROOT.iterdir(), sep='\n  ')
# Optional explicit paths. Keep None to use automatic discovery.
# RWF path must end at the folder containing train/ and val/.
RWF_INPUT_DIR = None  # Example: Path('/kaggle/input/datasets/<owner>/<rwf-slug>/rwf2000')
# Crop path can end at yolo11n or its parent dataset folder.
CROP_CACHE_INPUT = None  # Example: Path('/kaggle/input/datasets/<owner>/<crop-slug>/yolo11n')
# Optional override: exact Phase 5 fusion best.pt path.
BASELINE_FUSION_CHECKPOINT = None

if RWF_INPUT_DIR is not None:
    raw_root = Path(RWF_INPUT_DIR)
    if not raw_root.is_dir():
        raise FileNotFoundError(f'RWF_INPUT_DIR does not exist: {raw_root}')
    raw_candidates = [raw_root]
else:
    raw_candidates = sorted({
        p.parent.parent for p in INPUT_ROOT.rglob('Fight')
        if p.is_dir() and p.parent.name == 'train'
        and (p.parent.parent / 'val' / 'Fight').is_dir()
        and (p.parent.parent / 'train' / 'NonFight').is_dir()
    })
print('RWF roots found:', *raw_candidates, sep='\n  ')
if len(raw_candidates) != 1:
    raise RuntimeError('Attach exactly one RWF-2000 input or set RWF_INPUT_DIR to the folder containing train/ and val/.')
if not ((raw_candidates[0] / 'train' / 'Fight').is_dir() and
        (raw_candidates[0] / 'val' / 'Fight').is_dir() and
        (raw_candidates[0] / 'train' / 'NonFight').is_dir()):
    raise RuntimeError(f'RWF root must contain train/ and val/ with Fight/NonFight folders: {raw_candidates[0]}')
raw_target = REPO_DIR / 'data' / 'raw' / 'rwf2000'
raw_target.parent.mkdir(parents=True, exist_ok=True)
if raw_target.is_symlink():
    raw_target.unlink()
elif raw_target.exists():
    if any(raw_target.iterdir()):
        raise FileExistsError(f'Inspect existing non-empty path before replacing: {raw_target}')
    raw_target.rmdir()
raw_target.symlink_to(raw_candidates[0], target_is_directory=True)

manifest = pd.read_csv('data/splits/rwf2000.csv')
first_rows = manifest.groupby('split', sort=False).first().reset_index()
crop_search_root = Path(CROP_CACHE_INPUT) if CROP_CACHE_INPUT else INPUT_ROOT
if CROP_CACHE_INPUT and not crop_search_root.exists():
    raise FileNotFoundError(f'CROP_CACHE_INPUT does not exist: {crop_search_root}')
crop_files = list(crop_search_root.rglob('*.crop.npz'))
if not crop_files:
    crop_archives = []
    for archive_path in crop_search_root.rglob('*.zip'):
        try:
            with zipfile.ZipFile(archive_path) as archive:
                if any(name.lower().endswith('.crop.npz') for name in archive.namelist()):
                    crop_archives.append(archive_path)
        except zipfile.BadZipFile:
            continue
    if len(crop_archives) == 1:
        extract_root = Path('/kaggle/working/phase3_crop_cache_unpacked')
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(crop_archives[0]) as archive:
            for member in archive.infolist():
                target = (extract_root / member.filename).resolve()
                if target != extract_root.resolve() and extract_root.resolve() not in target.parents:
                    raise ValueError(f'Unsafe path in crop-cache archive: {member.filename}')
            archive.extractall(extract_root)
        crop_search_root = extract_root
        crop_files = list(crop_search_root.rglob('*.crop.npz'))
print('Crop archives discovered:', len(crop_files))

def crop_path_candidates(root, row):
    relative = Path(row['relative_path'])
    if relative.parts[:2] == ('data', 'raw'):
        relative = Path(*relative.parts[3:])
    filename = f'{relative.name}.crop.npz'
    return [root / row['dataset'] / row['split'] / relative.parent / filename,
            root / row['dataset'] / relative.parent / filename]

crop_roots = set()
for crop_file in crop_files:
    for part_index, part in enumerate(crop_file.parts):
        if part == 'rwf2000':
            crop_roots.add(Path(*crop_file.parts[:part_index]))
matching_roots, root_diagnostics = [], {}
for root in crop_roots:
    matched = sum(any(path.is_file() for path in crop_path_candidates(root, row))
                  for row in first_rows.to_dict('records'))
    root_diagnostics[str(root)] = f'{matched}/{len(first_rows)} split samples found'
    if matched == len(first_rows):
        matching_roots.append(root)
print('Crop cache candidates:', root_diagnostics)
if len(matching_roots) != 1:
    raise RuntimeError(f'Could not identify exactly one complete RWF crop-cache root. Candidate roots: {root_diagnostics}. Set CROP_CACHE_INPUT to the correct mounted folder.')
CROP_CACHE_DIR = matching_roots[0]
print('Crop cache root:', CROP_CACHE_DIR)

if BASELINE_FUSION_CHECKPOINT is None:
    baseline_candidates = [p for p in INPUT_ROOT.rglob('best.pt')
                           if 'x3d_s_rwf2000_late_fusion' in p.parts]
    if len(baseline_candidates) != 1:
        raise RuntimeError('Attach the Phase 5 output dataset or set BASELINE_FUSION_CHECKPOINT to its late-fusion best.pt path.')
    BASELINE_FUSION_CHECKPOINT = baseline_candidates[0]
else:
    BASELINE_FUSION_CHECKPOINT = Path(BASELINE_FUSION_CHECKPOINT)
if not BASELINE_FUSION_CHECKPOINT.is_file():
    raise FileNotFoundError(BASELINE_FUSION_CHECKPOINT)
print('Phase 5 baseline checkpoint:', BASELINE_FUSION_CHECKPOINT)


## 3. Build one reusable paired-view tensor cache and verify it

In [ ]:
TENSOR_CACHE = Path('/kaggle/temp/phase6_tensor_cache/rwf2000')
INPUT_TAG = 'rwf_fullcrop_f16_s160_tau2p00_m0p10_v1'
subprocess.run([sys.executable, 'scripts/cache_tensors.py',
    '--manifest', 'data/splits/rwf2000.csv',
    '--config', 'configs/preprocess_crops.yaml',
    '--model', 'x3d_m', '--cache-dir', str(TENSOR_CACHE),
    '--crop-cache-dir', str(CROP_CACHE_DIR), '--mode', 'both', '--workers', '2'], check=True)
REPORT_DIR = Path('/kaggle/working/experiments/phase6/cache_verification')
subprocess.run([sys.executable, 'scripts/verify_tensor_cache.py',
    '--cache-dir', str(TENSOR_CACHE), '--input-tag', INPUT_TAG,
    '--manifest', 'data/splits/rwf2000.csv', '--report-dir', str(REPORT_DIR)], check=True)
print((REPORT_DIR / 'verification_report.md').read_text())


## 4. Evaluate the Phase 5 X3D-S late-fusion baseline

In [ ]:
OUTPUT_ROOT = Path('/kaggle/working/experiments/phase6')
BASELINE_EVAL = OUTPUT_ROOT / 'evaluation' / 'x3d_s_late_fusion'
subprocess.run([sys.executable, 'src/violence_detection/evaluate.py',
    '--checkpoint', str(BASELINE_FUSION_CHECKPOINT),
    '--config', 'configs/train_x3d_fusion.yaml',
    '--cache-dir', str(TENSOR_CACHE), '--input-tag', INPUT_TAG, '--mode', 'both',
    '--include-crop-status', '--batch-size', '1', '--warmup-batches', '5',
    '--output-dir', str(BASELINE_EVAL), '--split', 'test'], check=True)
print((BASELINE_EVAL / 'test_results.md').read_text())
print((BASELINE_EVAL / 'inference_speed.json').read_text())


## 5. Train X3D-M late fusion on both paired views with T4 x2

In [ ]:
EXPERIMENTS = OUTPUT_ROOT
CHECKPOINT_ROOT = Path('/kaggle/working/model_checkpoints/phase6')
X3DM_OUTPUT = EXPERIMENTS / 'x3d_m_rwf2000_late_fusion'
X3DM_CHECKPOINT_DIR = CHECKPOINT_ROOT / 'x3d_m_rwf2000_late_fusion'

def launch_training(config, output_dir, checkpoint_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nnodes=1',
        f'--nproc_per_node={GPU_COUNT}', 'src/violence_detection/train.py',
        '--config', config, '--cache-dir', str(TENSOR_CACHE), '--output-dir', str(output_dir),
        '--checkpoint-dir', str(checkpoint_dir)]
    print('Launching:', command)
    subprocess.run(command, check=True)

launch_training('configs/train_x3d_m_fusion.yaml', X3DM_OUTPUT, X3DM_CHECKPOINT_DIR)

## 6. Evaluate X3D-M on the official held-out RWF-2000 test split

In [ ]:
X3DM_EVAL = OUTPUT_ROOT / 'evaluation' / 'x3d_m_late_fusion'
subprocess.run([sys.executable, 'src/violence_detection/evaluate.py',
    '--checkpoint', str(X3DM_CHECKPOINT_DIR / 'best.pt'),
    '--config', 'configs/train_x3d_m_fusion.yaml',
    '--cache-dir', str(TENSOR_CACHE), '--input-tag', INPUT_TAG, '--mode', 'both',
    '--include-crop-status', '--batch-size', '1', '--warmup-batches', '5',
    '--output-dir', str(X3DM_EVAL), '--split', 'test'], check=True)
print((X3DM_EVAL / 'test_results.md').read_text())
print((X3DM_EVAL / 'inference_speed.json').read_text())


## 7. Fine-tune Swin3D-T as the optional extra (enabled by default)

In [ ]:
RUN_SWIN3D = True  # Set False to skip optional Task 38 if session time is limited.
SWIN_OUTPUT = EXPERIMENTS / 'swin3d_t_rwf2000_paired_views'
SWIN_CHECKPOINT_DIR = CHECKPOINT_ROOT / 'swin3d_t_rwf2000_paired_views'
SWIN_EVAL = OUTPUT_ROOT / 'evaluation' / 'swin3d_t_paired_views'
if RUN_SWIN3D:
    launch_training('configs/train_swin3d_t.yaml', SWIN_OUTPUT, SWIN_CHECKPOINT_DIR)
    subprocess.run([sys.executable, 'src/violence_detection/evaluate.py',
        '--checkpoint', str(SWIN_CHECKPOINT_DIR / 'best.pt'),
        '--config', 'configs/train_swin3d_t.yaml',
        '--cache-dir', str(TENSOR_CACHE), '--input-tag', INPUT_TAG, '--mode', 'both',
        '--include-crop-status', '--batch-size', '1', '--warmup-batches', '5',
        '--output-dir', str(SWIN_EVAL), '--split', 'test'], check=True)
    print((SWIN_EVAL / 'test_results.md').read_text())
    print((SWIN_EVAL / 'inference_speed.json').read_text())
else:
    print('Swin3D-T skipped by RUN_SWIN3D=False.')


## 8. Compare accuracy and inference speed, then record the recommended classifier

In [ ]:
comparison_command = [sys.executable, 'scripts/compare_phase6_models.py',
    '--result', f'x3d_s_late_fusion={BASELINE_EVAL}',
    '--result', f'x3d_m_late_fusion={X3DM_EVAL}',
    '--output-dir', str(OUTPUT_ROOT / 'comparison'), '--f1-tolerance', '0.005']
if RUN_SWIN3D:
    comparison_command[comparison_command.index('--output-dir'):comparison_command.index('--output-dir')] = [
        '--result', f'swin3d_t_paired_views={SWIN_EVAL}']
subprocess.run(comparison_command, check=True)
print((OUTPUT_ROOT / 'comparison' / 'model_comparison.md').read_text())
print((OUTPUT_ROOT / 'comparison' / 'model_recommendation.json').read_text())


## 9. Output inventory

Download `/kaggle/working/experiments/phase6/` for histories, configs, evaluation results, and comparison reports. Download `/kaggle/working/model_checkpoints/phase6/` for `best.pt` and `last.pt`. Temporary video tensors are in `/kaggle/temp` and are not part of the saved notebook output.

In [ ]:
for root in [EXPERIMENTS, CHECKPOINT_ROOT]:
    print(f'\n{root}')
    if root.exists():
        for path in sorted(root.rglob('*')):
            if path.is_file():
                print(path, path.stat().st_size, 'bytes')